In [2]:
from __future__ import annotations

from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

raiz_busqueda = Path.cwd()
if not (raiz_busqueda / "3_automatizacion_e_ia").is_dir():
    raiz_busqueda = raiz_busqueda.parent

DATA_PATH = raiz_busqueda / "3_automatizacion_e_ia" / "Datos" / "ventas.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el fichero: {DATA_PATH}")

DATA_DIR = DATA_PATH.parent
ventas = pd.read_csv(DATA_PATH)

print("pandas:", pd.__version__)
print("Directorio de trabajo:", Path.cwd())
print("CSV utilizado:", DATA_PATH)
print("Datos existe:", DATA_PATH.exists())
print(ventas.head())

pandas: 3.0.5
Directorio de trabajo: c:\Users\alvar\Desktop\ICAI-MÁSTER\Desarrollo_de_Soluciones_de_IA\dsia-26-27-alvaro-poveda\Ejercicios
CSV utilizado: c:\Users\alvar\Desktop\ICAI-MÁSTER\Desarrollo_de_Soluciones_de_IA\dsia-26-27-alvaro-poveda\3_automatizacion_e_ia\Datos\ventas.csv
Datos existe: True
        fecha region   producto unidades  precio_unitario cliente_id
0  2026-01-09    Sur  Cable Kit        7            12.50       C005
1  2026-03-14  Oeste    Gateway       25           120.00       C032
2  2026-01-26  Oeste    Hub IoT        1            88.96       C016
3  2026-02-05  Norte    Gateway        0           120.00       C010
4  2026-02-07  Norte    Hub IoT       15            85.00       C027


In [6]:
print("Número de filas:", ventas.shape)
print("Tipos de datos de las columnas:", ventas.dtypes)
print("Número de valores nulos por columna:", ventas.isnull().sum())

Número de filas: (150, 6)
Tipos de datos de las columnas: fecha                  str
region                 str
producto               str
unidades               str
precio_unitario    float64
cliente_id             str
dtype: object
Número de valores nulos por columna: fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64


In [7]:
ventas.head()

,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027


In [4]:
def cargar_ventas(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No existe el fichero: {path}")
    return pd.read_csv(path)


raw = cargar_ventas(DATA_PATH)
raw

,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027
...,...,...,...,...,...,...
145,2026-02-15,Sur,Sensor A,16,19.50,C027
146,2026-01-13,Oeste,Hub IoT,19,85.00,C040
147,2026-02-09,Oeste,Sensor B,9,32.00,C009
148,2026-01-08,Oeste,Sensor B,5,32.31,C024


In [5]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = frame.copy()

    work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
    work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")

    ok = (
        work["unidades"].notna()
        & (work["unidades"] > 0)
        & work["precio_unitario"].notna()
        & (work["precio_unitario"] > 0)
    )

    validos = work.loc[ok].copy()
    errores = work.loc[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(raw)
print("Filas válidas:", len(validos))
print("Filas inválidas:", len(errores))

Filas válidas: 140
Filas inválidas: 10


In [ ]:
importe_por_region = (
    validos.groupby("region", as_index=False)["importe"].sum().sort_values("importe", ascending=False)
)
importe_por_region


,region,importe
0,Este,33122.53
1,Norte,22914.63
3,Sur,21396.35
2,Oeste,19164.66


In [9]:
top_productos = (
    validos.groupby("producto", as_index=False)["importe"].sum().sort_values("importe", ascending=False).head(3)
)
top_productos

,producto,importe
1,Gateway,40440.00
2,Hub IoT,33058.66
4,Sensor B,11568.66


In [10]:
compras_por_cliente = validos["cliente_id"].value_counts()
clientes_recurrentes = compras_por_cliente[compras_por_cliente > 1]
clientes_recurrentes

cliente_id
C027    9
C032    8
C004    8
C035    7
C013    6
C009    6
C005    5
C022    5
C038    5
C010    4
C018    4
C002    4
C040    4
C025    4
C015    4
C024    4
C023    4
C034    4
C037    4
C016    3
C003    3
C007    3
C011    3
C029    3
C033    3
C030    3
C028    2
C036    2
C021    2
C039    2
C031    2
C001    2
C020    2
C017    2
Name: count, dtype: int64

In [16]:
import json

salida_csv = DATA_DIR / "ventas_limpias.csv"
salida_json = DATA_DIR / "calidad_datos.json"

validos.to_csv(salida_csv, index=False)

calidad = {
    "filas_totales": int(len(raw)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
salida_json.write_text(json.dumps(calidad, indent=2, ensure_ascii=False), encoding="utf-8")

print("escrito:", salida_csv)
print("escrito:", salida_json)

calidad

escrito: c:\Users\alvar\Desktop\ICAI-MÁSTER\Desarrollo_de_Soluciones_de_IA\dsia-26-27-alvaro-poveda\3_automatizacion_e_ia\Datos\ventas_limpias.csv
escrito: c:\Users\alvar\Desktop\ICAI-MÁSTER\Desarrollo_de_Soluciones_de_IA\dsia-26-27-alvaro-poveda\3_automatizacion_e_ia\Datos\calidad_datos.json


{'filas_totales': 150,
 'filas_validas': 140,
 'filas_invalidas': 10,
 'importe_total': 96598.17000000001}